In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def run_decoder_bootstrapping(
    neural_data, trial_labels, subset_size_D, n_bootstraps=100, test_size=0.2
):
    """
    Bootstraps linear decoders on non-overlapping subsets of neurons.

    Parameters:
    -----------
    neural_data : np.ndarray
        Shape (m, n) where m is neurons and n is trials (as requested).
    trial_labels : np.ndarray
        Shape (1, n) or (n,). The class labels for the trials.
    subset_size_D : int
        Number of neurons to include in each of the two non-overlapping sets.
    n_bootstraps : int
        Number of times to resample neurons and retrain.
    test_size : float
        Fraction of data to use for testing (0.0 to 1.0).

    Returns:
    --------
    list of dicts
        Each dict contains:
        - 'probs_A': (n_test_samples, n_classes) probabilities from Set A
        - 'probs_B': (n_test_samples, n_classes) probabilities from Set B
        - 'y_test': Actual labels for the test set (needed for PID)
        - 'neurons_A_indices': Indices of neurons used for Set A
        - 'neurons_B_indices': Indices of neurons used for Set B
    """

    # 1. Shape Handling
    # The prompt specifies neurons (m x n) and trials (1 x n).
    # Scikit-learn expects (n_samples, n_features).
    # We transpose: (m, n) -> (n, m)
    X = neural_data.T

    # Ensure labels are 1D array
    y = trial_labels.flatten()

    n_trials, n_neurons = X.shape

    # Check constraints
    if 2 * subset_size_D > n_neurons:
        raise ValueError(
            f"Cannot select 2 non-overlapping sets of size {subset_size_D} "
            f"from {n_neurons} neurons."
        )

    results = []

    print(f"Starting bootstrapping: {n_bootstraps} iterations...")
    print(f"Total Neurons: {n_neurons}, Subset Size (D): {subset_size_D}")

    for i in range(n_bootstraps):
        # 2. Sub-sample neurons
        # Random permutation of all neuron indices
        permuted_indices = np.random.permutation(n_neurons)

        # Select first D as Set A, next D as Set B (guaranteed non-overlapping)
        idx_A = permuted_indices[:subset_size_D]
        idx_B = permuted_indices[subset_size_D : 2 * subset_size_D]

        # Slice the data
        X_subset_A = X[:, idx_A]
        X_subset_B = X[:, idx_B]

        # 3. Train-Test Split
        # Critical: We must use the same random state or split logic for both A and B
        # so that the rows in X_test_A correspond to the same trials as X_test_B.
        # We split the INDICES, then apply to both.
        indices = np.arange(n_trials)
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            stratify=y,  # Good practice to keep class ratios consistent
            random_state=None,  # None means random every bootstrap iteration
        )

        # Prepare datasets
        X_train_A, X_test_A = X_subset_A[train_idx], X_subset_A[test_idx]
        X_train_B, X_test_B = X_subset_B[train_idx], X_subset_B[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # 4. Train Decoders
        # Using Logistic Regression as the linear decoder
        # Solver 'liblinear' is good for smaller datasets, 'lbfgs' for larger
        clf_A = LogisticRegression(solver="lbfgs", max_iter=1000)
        clf_B = LogisticRegression(solver="lbfgs", max_iter=1000)

        clf_A.fit(X_train_A, y_train)
        clf_B.fit(X_train_B, y_train)

        # 5. Store Output (Probabilities)
        # predict_proba returns [P(class_0), P(class_1), ...]
        probs_A = clf_A.predict_proba(X_test_A)
        probs_B = clf_B.predict_proba(X_test_B)

        # Save results for this iteration
        run_data = {
            "iteration": i,
            "probs_A": probs_A,
            "probs_B": probs_B,
            "y_test": y_test,  # Ground truth for this split
            "neurons_A_indices": idx_A,
            "neurons_B_indices": idx_B,
        }
        results.append(run_data)

    print("Bootstrapping complete.")
    return results


# ==========================================
# Example Usage
# ==========================================
if __name__ == "__main__":
    # Generate Synthetic Data
    # 500 neurons (m), 200 trials (n)
    m_neurons = 500
    n_trials = 200

    # Create random neural activity
    neural_data = np.random.randn(m_neurons, n_trials)

    # Create binary labels (0 or 1)
    labels = np.random.randint(0, 2, size=(1, n_trials))

    # Add some signal to a subset of neurons so the decoder actually works
    # (Neurons 0-50 prefer class 1)
    signal_mask = labels.flatten() == 1
    neural_data[:50, signal_mask] += 0.5

    # Run the pipeline
    # We want sets of 50 neurons, 20 bootstraps
    D = 50
    bootstrap_results = run_decoder_bootstrapping(
        neural_data, labels, subset_size_D=D, n_bootstraps=20
    )

    # Inspect the first result
    first_run = bootstrap_results[0]

    print("\n--- Output Inspection (Run 0) ---")
    print(
        f"Probabilities A shape: {first_run['probs_A'].shape}"
    )  # Should be (n_test_trials, n_classes)
    print(f"Probabilities B shape: {first_run['probs_B'].shape}")
    print(f"Ground Truth shape:    {first_run['y_test'].shape}")

    print("\nExample Probabilities (First 5 test trials, Decoder A):")
    print(first_run["probs_A"][:5])

    print("\nNote for PID:")
    print("You now have P(Response|NeuronSetA) and P(Response|NeuronSetB).")
    print("You can calculate Mutual Information I(Stimulus; DecoderA) and I(Stimulus; DecoderB)")
    print("to proceed with Synergy/Redundancy calculations.")

In [4]:
first_run.keys()

dict_keys(['iteration', 'probs_A', 'probs_B', 'y_test', 'neurons_A_indices', 'neurons_B_indices'])

In [ ]:
def run_decoder_bootstrapping(
    neural_data,
    trial_labels,
    subset_size_D,
    n_bootstraps=100,
    test_size=0.2,
    congruent_mask=None,
    incongruent_mask=None,
):
    """
    Bootstraps linear decoders on non-overlapping subsets of neurons.

    Parameters:
    -----------
    neural_data : np.ndarray
        Shape (m, n) where m is neurons and n is trials (as requested).
    trial_labels : np.ndarray
        Shape (1, n) or (n,). The class labels for the trials.
    subset_size_D : int
        Number of neurons to include in each of the two non-overlapping sets.
    n_bootstraps : int
        Number of times to resample neurons and retrain.
    test_size : float
        Fraction of data to use for testing (0.0 to 1.0).
    congruent_mask : np.ndarray, optional
        Boolean mask of shape (1, n) or (n,) indicating congruent trials.
    incongruent_mask : np.ndarray, optional
        Boolean mask of shape (1, n) or (n,) indicating incongruent trials.

    Returns:
    --------
    list of dicts
        Each dict contains:
        - 'probs_A': (n_test_samples, n_classes) probabilities from Set A
        - 'probs_B': (n_test_samples, n_classes) probabilities from Set B
        - 'y_test': Actual labels for the test set (needed for PID)
        - 'neurons_A_indices': Indices of neurons used for Set A
        - 'neurons_B_indices': Indices of neurons used for Set B
    """

    # 1. Shape Handling
    # The prompt specifies neurons (m x n) and trials (1 x n).
    # Scikit-learn expects (n_samples, n_features).
    # We transpose: (m, n) -> (n, m)
    X = neural_data.T

    # Ensure labels are 1D array
    y = trial_labels.flatten()

    # Flatten masks if provided to match y shape
    if congruent_mask is not None:
        congruent_mask = np.array(congruent_mask).flatten()
    if incongruent_mask is not None:
        incongruent_mask = np.array(incongruent_mask).flatten()

    n_trials, n_neurons = X.shape

    # Check constraints
    if 2 * subset_size_D > n_neurons:
        raise ValueError(
            f"Cannot select 2 non-overlapping sets of size {subset_size_D} "
            f"from {n_neurons} neurons."
        )

    results = []

    print(f"Starting bootstrapping: {n_bootstraps} iterations...")
    print(f"Total Neurons: {n_neurons}, Subset Size (D): {subset_size_D}")

    for i in range(n_bootstraps):
        # 2. Sub-sample neurons
        # Random permutation of all neuron indices
        permuted_indices = np.random.permutation(n_neurons)

        # Select first D as Set A, next D as Set B (guaranteed non-overlapping)
        idx_A = permuted_indices[:subset_size_D]
        idx_B = permuted_indices[subset_size_D : 2 * subset_size_D]

        # Slice the data
        X_subset_A = X[:, idx_A]
        X_subset_B = X[:, idx_B]

        # 3. Train-Test Split
        # Critical: We must use the same random state or split logic for both A and B
        # so that the rows in X_test_A correspond to the same trials as X_test_B.
        # We split the INDICES, then apply to both.
        indices = np.arange(n_trials)
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            stratify=y,  # Good practice to keep class ratios consistent
            random_state=None,  # None means random every bootstrap iteration
        )

        # Prepare datasets
        X_train_A, X_test_A = X_subset_A[train_idx], X_subset_A[test_idx]
        X_train_B, X_test_B = X_subset_B[train_idx], X_subset_B[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # 4. Train Decoders
        # Using Logistic Regression as the linear decoder
        # Solver 'liblinear' is good for smaller datasets, 'lbfgs' for larger
        clf_A = LogisticRegression(solver="lbfgs", max_iter=1000)
        clf_B = LogisticRegression(solver="lbfgs", max_iter=1000)

        clf_A.fit(X_train_A, y_train)
        clf_B.fit(X_train_B, y_train)

        # 5. Store Output (Probabilities)
        # predict_proba returns [P(class_0), P(class_1), ...]
        probs_A = clf_A.predict_proba(X_test_A)
        probs_B = clf_B.predict_proba(X_test_B)

        # Save results for this iteration
        run_data = {
            "iteration": i,
            "probs_A": probs_A,
            "probs_B": probs_B,
            "y_test": y_test,  # Ground truth for this split
            "neurons_A_indices": idx_A,
            "neurons_B_indices": idx_B,
        }

        # If masks are provided, subset the TEST results
        if congruent_mask is not None and incongruent_mask is not None:
            # We must apply the same test_idx to the masks to find out which
            # of the *test* items are congruent/incongruent
            test_mask_cong = congruent_mask[test_idx]
            test_mask_incong = incongruent_mask[test_idx]

            # Subset the probabilities and labels
            run_data["probs_A_cong"] = probs_A[test_mask_cong]
            run_data["probs_A_incong"] = probs_A[test_mask_incong]

            run_data["probs_B_cong"] = probs_B[test_mask_cong]
            run_data["probs_B_incong"] = probs_B[test_mask_incong]

            run_data["y_test_cong"] = y_test[test_mask_cong]
            run_data["y_test_incong"] = y_test[test_mask_incong]

        results.append(run_data)

    print("Bootstrapping complete.")
    return results


# ==========================================
# Example Usage
# ==========================================
if __name__ == "__main__":
    # Generate Synthetic Data
    # 500 neurons (m), 200 trials (n)
    m_neurons = 500
    n_trials = 200

    # Create random neural activity
    neural_data = np.random.randn(m_neurons, n_trials)

    # Create binary labels (0 or 1)
    labels = np.random.randint(0, 2, size=(1, n_trials))

    # Add some signal to a subset of neurons so the decoder actually works
    # (Neurons 0-50 prefer class 1)
    signal_mask = labels.flatten() == 1
    neural_data[:50, signal_mask] += 0.5

    # Create synthetic congruent/incongruent masks
    # E.g., first 100 are congruent, next 100 are incongruent
    cong_mask = np.zeros(n_trials, dtype=bool)
    cong_mask[:100] = True
    incong_mask = ~cong_mask

    # Run the pipeline
    # We want sets of 50 neurons, 20 bootstraps
    D = 50
    bootstrap_results = run_decoder_bootstrapping(
        neural_data,
        labels,
        subset_size_D=D,
        n_bootstraps=20,
        congruent_mask=cong_mask,
        incongruent_mask=incong_mask,
    )

    # Inspect the first result
    first_run = bootstrap_results[0]

    print("\n--- Output Inspection (Run 0) ---")
    print(
        f"Probabilities A shape: {first_run['probs_A'].shape}"
    )  # Should be (n_test_trials, n_classes)
    print(f"Probabilities B shape: {first_run['probs_B'].shape}")
    print(f"Ground Truth shape:    {first_run['y_test'].shape}")

    # Verify the splitting worked
    if "probs_A_cong" in first_run:
        n_cong_test = first_run["probs_A_cong"].shape[0]
        n_incong_test = first_run["probs_A_incong"].shape[0]
        print(f"\n--- Condition Splitting ---")
        print(f"Congruent test samples:   {n_cong_test}")
        print(f"Incongruent test samples: {n_incong_test}")
        print(f"Total test samples:       {n_cong_test + n_incong_test}")

    print("\nExample Probabilities (First 5 test trials, Decoder A):")
    print(first_run["probs_A"][:5])

    print("\nNote for PID:")
    print("You now have P(Response|NeuronSetA) and P(Response|NeuronSetB).")
    print("You can calculate Mutual Information I(Stimulus; DecoderA) and I(Stimulus; DecoderB)")
    print("to proceed with Synergy/Redundancy calculations.")